# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets available in the dataset with their @id
record_set_objs = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_set_objs)}\n")
for idx, rec in enumerate(record_set_objs):
    print(f"[{idx}] Record Set @id: {rec['@id']}")
    print(f"    Name: {rec['name'] if 'name' in rec else 'N/A'}")
    print(f"    Description: {rec['description'] if 'description' in rec else 'N/A'}")
    # List fields in this record set
    fields = rec.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"    Fields (@id):")
    for f in fields:
        print(f"        - {f['@id'] if isinstance(f, dict) and '@id' in f else f}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose record sets for extraction
# You may change these based on the record set @id output above.
record_sets = [
    'cr:RecordSet-Table1',    # Example: replace with the real @id output from previous cell
    'cr:RecordSet-Table2'     # If more available, add here
]

dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set: {record_set_id} ({len(dataframes[record_set_id])} rows)")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Explore columns of one record set
example_record_set = record_sets[0]
if example_record_set in dataframes:
    print(f"\nFields (@id) for '{example_record_set}':\n{dataframes[example_record_set].columns.tolist()}")
    display(dataframes[example_record_set].head())
else:
    print(f"Record set {example_record_set} not available in dataframes.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a record set and numeric field for EDA (use @id as shown in earlier overview)
record_set_id = record_sets[0] # Use the main table
numeric_field = 'cr:Field-abundance' # Example: replace with actual numeric @id from your schema

if record_set_id in dataframes and numeric_field in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]

    # Filter for high abundance (e.g., abundance > 10)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records in '{record_set_id}' with {numeric_field} > {threshold} (rows: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Optionally group by a categorical field, e.g., 'cr:Field-gene'
    group_field = 'cr:Field-gene' # Example: replace with real @id of a categorical field
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (showing means):")
        display(grouped_df.head())
    else:
        print(f"Field '{group_field}' not available for grouping.")
else:
    print(f"Either DataFrame for '{record_set_id}' or field '{numeric_field}' not found. Please update variable names to @id values from section 2.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if record_set_id in dataframes and numeric_field in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot of abundance vs another numeric field
    numeric_field2 = 'cr:Field-molecular_weight'  # Example: Use the @id from your schema
    if numeric_field2 in df.columns:
        plt.figure(figsize=(6, 4))
        plt.scatter(df[numeric_field2], df[numeric_field], alpha=0.6)
        plt.xlabel(numeric_field2)
        plt.ylabel(numeric_field)
        plt.title(f"{numeric_field} vs. {numeric_field2}")
        plt.show()
    else:
        print(f"Numeric field '{numeric_field2}' for scatter plot not found. Update its value as needed.")
else:
    print(f"Visualization could not be created. Ensure correct field/record set @id values.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*This notebook demonstrated how to load and analyze the Mass Spectrometry protein dataset using the `mlcroissant` library. We reviewed the available record sets, explored sample fields referenced by their Croissant `@id`, applied filtering and normalization, and visualized distributions. Modify the `@id` field names above based on the exact schema overview for deeper, domain-specific exploration!*